In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import SpectralCoclustering, KMeans
import warnings

# Suppress warnings
warnings.filterwarnings("ignore", category=UserWarning, module="pandas")

#  Load and preprocess the dataset (Exact preprocessing retained)
def load_and_preprocess_data(filepath):
    try:
        df = pd.read_excel(filepath, engine="odf")
    except Exception as e:
        raise ValueError(f"Error reading the file: {e}")

    print("Columns in the dataset:", df.columns)

    if 'FECHA_HORA' not in df.columns:
        raise KeyError("The column 'FECHA_HORA' does not exist in the dataset.")

    df['FECHA_HORA'] = pd.to_datetime(df['FECHA_HORA'], format='%d/%m/%Y %H:%M', errors='coerce')
    df.set_index('FECHA_HORA', inplace=True)

    df = df.apply(pd.to_numeric, errors='coerce')
    df.dropna(inplace=True)

    target_col = 'TORNEO-O3-AT_IN'
    if target_col not in df.columns:
        raise KeyError(f"The target column '{target_col}' does not exist in the dataset.")

    #  Add rolling calculations (exactly as requested)
    rolling_window = 24
    df[f'{target_col}_rolling_mean'] = df[target_col].rolling(window=rolling_window).mean()
    df[f'{target_col}_rolling_std'] = df[target_col].rolling(window=rolling_window).std()
    df[f'{target_col}_rolling_skew'] = df[target_col].rolling(window=rolling_window).skew()

    #  Add lagged features (exactly as requested)
    lags = 24
    for i in range(1, lags + 1):
        df[f'{target_col}_lag{i}'] = df[target_col].shift(i)

    df.dropna(inplace=True)  # Ensure no NaN values
    return df, target_col

#  Find optimal number of clusters using the Elbow method
def find_optimal_clusters(data, max_clusters=10):
    ssd = []
    for n_clusters in range(1, max_clusters + 1):
        kmeans = KMeans(n_clusters=n_clusters, random_state=42)
        kmeans.fit(data)
        ssd.append(kmeans.inertia_)

    plt.figure(figsize=(8, 5))
    plt.plot(range(1, max_clusters + 1), ssd, marker='o', linestyle='--')
    plt.xlabel("Number of Clusters")
    plt.ylabel("Sum of Squared Distances (SSD)")
    plt.title("Elbow Method for Optimal Number of Clusters")
    plt.grid()
    plt.show()

    # Automatically find the elbow point
    optimal_clusters = np.diff(ssd, 2).argmin() + 2  # Second derivative method
    print(f"Optimal number of clusters: {optimal_clusters}")
    return optimal_clusters

#  Save each co-cluster to a separate .ods file including the target column
def save_coclusters(df, feature_clusters, instance_clusters, target_col, output_dir="cocluster_outputs"):
    os.makedirs(output_dir, exist_ok=True)  # Ensure directory exists

    for row_cluster_id in range(len(instance_clusters)):
        for col_cluster_id in range(len(feature_clusters)):
            feature_names = feature_clusters[col_cluster_id]

            # Get row indices and feature subset
            row_indices = instance_clusters[row_cluster_id]
            cocluster_df = df[feature_names].iloc[row_indices]

            #  Include the target variable
            cocluster_df[target_col] = df[target_col].iloc[row_indices]

            #  Save to file
            filename = os.path.join(output_dir, f"cocluster_{row_cluster_id}_{col_cluster_id}.ods")
            cocluster_df.to_excel(filename, engine="odf")
            print(f"Saved: {filename}")

#  Main function
def main():
    filepath = r"E:\Abroad period research\Time series forecasting\Torneo_2015_2023\Torneo_2015_2023.ods"
    try:
        #  Load and preprocess data (including lagging & rolling statistics)
        df, target_col = load_and_preprocess_data(filepath)
        print("Data loaded and cleaned.")

        #  Select only numeric features (excluding the target column)
        numeric_features = df.select_dtypes(include=[np.number]).drop(columns=[target_col])

        #  Find optimal clusters
        optimal_clusters = find_optimal_clusters(numeric_features)

        print("Applying Spectral Co-Clustering...")
        bicluster = SpectralCoclustering(n_clusters=optimal_clusters, random_state=42)
        bicluster.fit(numeric_features)

        #  Extract feature clusters
        feature_clusters = {i: [] for i in range(optimal_clusters)}
        for i, cluster in enumerate(bicluster.column_labels_):
            feature_clusters[cluster].append(numeric_features.columns[i])

        #  Extract row clusters
        instance_clusters = {i: [] for i in range(optimal_clusters)}
        for i, cluster in enumerate(bicluster.row_labels_):
            instance_clusters[cluster].append(i)

        #  Save co-clustered data
        save_coclusters(df, feature_clusters, instance_clusters, target_col)

    except Exception as e:
        print(f"Error: {e}")

if __name__ == "__main__":
    main()
